# LeetCode 1339 — Maximum Product of Splitted Binary Tree bằng Python

Link đề: https://leetcode.com/problems/maximum-product-of-splitted-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def maxProduct(self, root: TreeNode | None) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra: cắt **một cạnh** ⇔ tách ra **một cây con** (cây con của nút con ở đầu dưới cạnh); phần còn lại có tổng `total − sub`.
2. Thành thạo kỹ thuật **tính tổng mọi cây con bằng DFS hậu thứ tự** trong một lượt `O(n)`.
3. Viết 4 cách: brute force cắt từng cạnh `O(n²)` → hai lượt DFS → một lượt DFS + danh sách tổng → hậu thứ tự **bằng ngăn xếp** (an toàn với cây sâu 5·10⁴ nút).
4. Hiểu yêu cầu "**lấy max trước rồi mới mod** `10^9 + 7`" và vì sao Python (số nguyên không giới hạn) giúp làm đúng dễ dàng.
5. Tránh lỗi: mod trước khi so sánh, chỉ cắt cạnh kề gốc, chỉ xét tổng `≤ total/2`, đệ quy quá sâu.

## 2. Đề bài & ví dụ

Cho gốc `root` của cây nhị phân. Xoá **một cạnh** để tách cây thành **hai cây con** sao cho **tích tổng giá trị** của hai cây là lớn nhất. Trả về tích lớn nhất đó **modulo `10^9 + 7`**.

*Lưu ý:* phải **tối đa hoá tích trước**, rồi mới lấy mod — không phải tối đa hoá phần dư sau khi mod.

**Ví dụ 1:** `root = [1,2,3,4,5,6]` → `110` (cắt cạnh 1–2: hai cây có tổng 11 và 10, tích 110).

**Ví dụ 2:** `root = [1,null,2,3,4,null,null,5,6]` → `90` (tổng 15 và 6).

**Ràng buộc:** số nút trong `[2, 5·10^4]`; `1 <= Node.val <= 10^4`.

In [1]:
import random
import sys
import time
from collections import deque
from typing import List, Optional

MOD = 10**9 + 7

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(arr):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = không có nút)."""
    if not arr or arr[0] is None:
        return None
    root = TreeNode(arr[0])
    q = deque([root])
    i = 1
    while q and i < len(arr):
        node = q.popleft()
        if i < len(arr) and arr[i] is not None:
            node.left = TreeNode(arr[i])
            q.append(node.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            node.right = TreeNode(arr[i])
            q.append(node.right)
        i += 1
    return root

def tree_to_list(root):
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        q.append(node.left)
        q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

examples = [([1, 2, 3, 4, 5, 6], 110), ([1, None, 2, 3, 4, None, None, 5, 6], 90)]
for arr, want in examples:
    print(arr, "-> mong đợi", want, "| dựng lại:", tree_to_list(build_tree(arr)))

[1, 2, 3, 4, 5, 6] -> mong đợi 110 | dựng lại: [1, 2, 3, 4, 5, 6]
[1, None, 2, 3, 4, None, None, 5, 6] -> mong đợi 90 | dựng lại: [1, None, 2, 3, 4, None, None, 5, 6]


## 3. Kiến thức nền

### 3.1 Cây nhị phân, `TreeNode` và hàm vẽ cây

Ta tự dựng `TreeNode` + `build_tree` (level-order có `None`) như LeetCode. Hàm `show` in cây thụt lề, kèm **tổng cây con** của mỗi nút.

In [2]:
def subtree_sum(node):
    return 0 if node is None else node.val + subtree_sum(node.left) + subtree_sum(node.right)

def show(root, indent=0, tag="root"):
    if root is None:
        return
    print("    " * indent + f"{tag}: {root.val}  (tổng cây con = {subtree_sum(root)})")
    show(root.left, indent + 1, "L")
    show(root.right, indent + 1, "R")

show(build_tree(examples[0][0]))

root: 1  (tổng cây con = 21)
    L: 2  (tổng cây con = 11)
        L: 4  (tổng cây con = 4)
        R: 5  (tổng cây con = 5)
    R: 3  (tổng cây con = 9)
        L: 6  (tổng cây con = 6)


### 3.2 Tổng cây con bằng đệ quy hậu thứ tự

`sum(node) = node.val + sum(node.left) + sum(node.right)`, với `sum(None) = 0`. Tính con **trước**, rồi cộng vào cha ⇒ mỗi nút được tính **đúng một lần** nếu ta lưu lại kết quả trong quá trình duyệt (thay vì gọi lại `subtree_sum` cho từng nút như hàm `show` ở trên — cách đó tốn `O(n²)`).

In [3]:
def all_subtree_sums(root):
    sums = []
    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        sums.append(s)
        return s
    total = dfs(root)
    return total, sums

print(all_subtree_sums(build_tree(examples[0][0])))

(21, [4, 5, 11, 6, 9, 21])


### 3.3 Số nguyên lớn và phép mod trong Python

Python có số nguyên **không giới hạn** ⇒ tích `s·(total − s)` (tối đa khoảng `(2.5·10^8)^2 ≈ 6·10^16`) được tính **chính xác**, không tràn số như `int` 32-bit trong C++/Java. Vì thế ta cứ tìm `max` trên số thật rồi `% MOD` một lần cuối.

In [4]:
total = 5 * 10**4 * 10**4            # tổng lớn nhất có thể: 5·10^8
best = (total // 2) * (total - total // 2)
print("tích lớn nhất có thể ≈", best, "| > 2^31?", best > 2**31, "| mod =", best % MOD)

tích lớn nhất có thể ≈ 62500000000000000 | > 2^31? True | mod = 562500007


### 3.4 Hậu thứ tự không đệ quy (ngăn xếp)

Cây có thể lệch (5·10⁴ nút thẳng hàng) ⇒ đệ quy sâu 5·10⁴ tầng vượt giới hạn mặc định ~1000 của Python. Cách đơn giản để có hậu thứ tự bằng ngăn xếp: duyệt **tiền thứ tự** (gốc trước) lưu vào danh sách `order`, rồi xử lý `order` theo **thứ tự ngược** — khi đó mọi con được xử lý trước cha.

In [5]:
def postorder_iter(root):
    order, stack = [], [root]
    while stack:
        node = stack.pop()
        if node:
            order.append(node)
            stack.append(node.left)
            stack.append(node.right)
    return [nd.val for nd in reversed(order)]

print("hậu thứ tự (ngăn xếp):", postorder_iter(build_tree([1, 2, 3, 4, 5, 6])))

hậu thứ tự (ngăn xếp): [4, 5, 2, 6, 3, 1]


### 3.5 Tích của hai số có tổng cố định

Với `a + b = T` cố định, tích `a·b = a·(T − a)` lớn nhất khi `a` **gần `T/2` nhất** (parabol úp). Đây là trực giác: muốn tích lớn ⇒ cắt sao cho hai phần **cân bằng** nhất.

In [6]:
T = 21
print([(a, a * (T - a)) for a in range(0, T + 1, 3)])

[(0, 0), (3, 54), (6, 90), (9, 108), (12, 108), (15, 90), (18, 54), (21, 0)]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** `[1,2,3,4,5,6]`, tổng toàn cây `T = 21`:

```
        1
      /   \
     2     3
    / \   /
   4   5 6
```

Mỗi cạnh nối cha với một con `c`; cắt cạnh đó ⇒ một phần là **cây con gốc `c`** (tổng `s`), phần kia tổng `21 − s`.

| cạnh cắt | s (cây con dưới) | 21 − s | tích |
|---|---|---|---|
| 1–2 | 2+4+5 = 11 | 10 | **110** |
| 1–3 | 3+6 = 9 | 12 | 108 |
| 2–4 | 4 | 17 | 68 |
| 2–5 | 5 | 16 | 80 |
| 3–6 | 6 | 15 | 90 |

Lớn nhất: `110` ✔. Nhận xét: chỉ cần **tổng mọi cây con** + **tổng toàn cây** là đủ.

In [7]:
total, sums = all_subtree_sums(build_tree(examples[0][0]))
print("T =", total)
for s in sums:
    print(f"s = {s:2}, T - s = {total - s:2}, tích = {s * (total - s)}")

T = 21
s =  4, T - s = 17, tích = 68
s =  5, T - s = 16, tích = 80
s = 11, T - s = 10, tích = 110
s =  6, T - s = 15, tích = 90
s =  9, T - s = 12, tích = 108
s = 21, T - s =  0, tích = 0


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** làm đúng như đề: với **mỗi cạnh** `(cha, con)`, tạm thời **cắt** liên kết, tính lại tổng hai cây bằng hai lượt duyệt đầy đủ, ghi nhận tích, rồi **nối lại**. Có `n − 1` cạnh, mỗi lần tính tổng tốn `O(n)`.

In [8]:
def max_product_brute(root):
    def tree_sum(node):
        total, stack = 0, [node]
        while stack:
            nd = stack.pop()
            if nd:
                total += nd.val
                stack.append(nd.left)
                stack.append(nd.right)
        return total

    best = 0
    stack = [root]
    while stack:
        parent = stack.pop()
        for side in ("left", "right"):
            child = getattr(parent, side)
            if child is None:
                continue
            setattr(parent, side, None)              # cắt cạnh
            prod = tree_sum(root) * tree_sum(child)
            setattr(parent, side, child)             # nối lại
            best = max(best, prod)
            stack.append(child)
    return best % MOD

for arr, want in examples:
    print(max_product_brute(build_tree(arr)), "| mong đợi", want)

110 | mong đợi 110
90 | mong đợi 90


**Độ phức tạp:** `O(n²)` thời gian (`n = 5·10^4` ⇒ `2.5·10^9` bước — quá chậm), `O(n)` bộ nhớ ngăn xếp.

**Nút thắt:** tổng của mỗi cây con bị tính lại rất nhiều lần. Nhưng tổng cây con có cấu trúc đệ quy: `sum(node) = val + sum(left) + sum(right)` ⇒ tính **tất cả** trong **một** lượt hậu thứ tự. Tổng phần còn lại thì không cần duyệt: chính là `total − sum(child)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** các cạnh tương ứng 1–1 với các nút **không phải gốc** (cạnh nối nút đó với cha). Cắt cạnh trên nút `v` ⇒ hai phần có tổng `sub(v)` và `total − sub(v)`.

**Insight 2:** `total = sub(root)`, và mọi `sub(v)` tính được trong một lượt DFS hậu thứ tự.

**Kế hoạch:**
1. Lượt 1: tính `total`.
2. Lượt 2: DFS hậu thứ tự, tại mỗi nút tính `s = sub(v)` và cập nhật `best = max(best, s·(total − s))`.
3. Trả `best % MOD`.

Ở gốc, `s = total` ⇒ tích `0` — không ảnh hưởng `max`, nên không cần loại riêng.

**Quy tắc "nếu… thì…":** nếu `s·(total − s) > best` thì cập nhật. Chỉ mod **sau cùng**.

**Biến thể tối ưu nhỏ:** vì tích lớn nhất khi `s` gần `total/2`, có thể chỉ tìm `s` có `|2s − total|` nhỏ nhất rồi tính tích một lần (Cách 3 dùng cả hai cách để so sánh).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai lượt DFS đệ quy

**Từng đoạn:**
1. `total = dfs_sum(root)`.
2. `dfs(node)` trả về tổng cây con, đồng thời cập nhật `best` (biến `nonlocal`).
3. `return best % MOD`.

In [9]:
def max_product_two_pass(root):
    def dfs_sum(node):
        return 0 if node is None else node.val + dfs_sum(node.left) + dfs_sum(node.right)

    total = dfs_sum(root)
    best = 0

    def dfs(node):
        nonlocal best
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        best = max(best, s * (total - s))
        return s

    dfs(root)
    return best % MOD

for arr, want in examples:
    print(max_product_two_pass(build_tree(arr)), "| mong đợi", want)

110 | mong đợi 110
90 | mong đợi 90


Code nộp LeetCode (Cách 2):

In [10]:
class SolutionTwoPass:
    def maxProduct(self, root: Optional[TreeNode]) -> int:
        def dfs_sum(node):
            return 0 if node is None else node.val + dfs_sum(node.left) + dfs_sum(node.right)

        total = dfs_sum(root)
        best = 0

        def dfs(node):
            nonlocal best
            if node is None:
                return 0
            s = node.val + dfs(node.left) + dfs(node.right)
            if s * (total - s) > best:
                best = s * (total - s)
            return s

        dfs(root)
        return best % (10**9 + 7)

print([SolutionTwoPass().maxProduct(build_tree(a)) for a, _ in examples])

[110, 90]


**Độ phức tạp:** `O(n)` thời gian (hai lượt), `O(h)` bộ nhớ đệ quy (`h` = chiều cao, xấu nhất `n`).

### 7.2 Cách 3 — Một lượt DFS, lưu mọi tổng cây con vào danh sách

Lượt DFS duy nhất trả về `total` và ghi mọi `sub(v)` vào list `sums`. Sau đó `max(s * (total − s) for s in sums)`. Ưu điểm: không cần biết `total` khi đang duyệt; dễ mở rộng (ví dụ tìm `s` gần `total/2` nhất).

In [11]:
def max_product_one_pass(root):
    sums = []

    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        sums.append(s)
        return s

    total = dfs(root)
    return max(s * (total - s) for s in sums) % MOD

def max_product_closest_half(root):
    sums = []

    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        sums.append(s)
        return s

    total = dfs(root)
    s = min(sums, key=lambda x: abs(2 * x - total))       # gần total/2 nhất
    return s * (total - s) % MOD

for arr, want in examples:
    r = build_tree(arr)
    print(max_product_one_pass(r), max_product_closest_half(r), "| mong đợi", want)

110 110 | mong đợi 110
90 90 | mong đợi 90


Code nộp LeetCode (Cách 3):

In [12]:
class SolutionOnePass:
    def maxProduct(self, root: Optional[TreeNode]) -> int:
        sums = []

        def dfs(node):
            if node is None:
                return 0
            s = node.val + dfs(node.left) + dfs(node.right)
            sums.append(s)
            return s

        total = dfs(root)
        return max(s * (total - s) for s in sums) % (10**9 + 7)

print([SolutionOnePass().maxProduct(build_tree(a)) for a, _ in examples])

[110, 90]


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (danh sách `sums` + đệ quy).

### 7.3 Cách 4 — Hậu thứ tự bằng ngăn xếp (không đệ quy)

**Từng đoạn:**
1. Duyệt tiền thứ tự bằng ngăn xếp, ghi các nút vào `order`.
2. Duyệt `order` **ngược**: `sub[node] = node.val + sub[left] + sub[right]` (con đã có trước cha). Lưu `sub` trong `dict` theo nút.
3. `total = sub[root]`; `best = max(s·(total − s))`.

An toàn với cây lệch 5·10⁴ nút, không phụ thuộc giới hạn đệ quy.

In [13]:
def max_product_iter(root):
    order, stack = [], [root]
    while stack:
        node = stack.pop()
        if node:
            order.append(node)
            stack.append(node.left)
            stack.append(node.right)
    sub = {None: 0}
    for node in reversed(order):
        sub[node] = node.val + sub[node.left] + sub[node.right]
    total = sub[root]
    best = max(sub[nd] * (total - sub[nd]) for nd in order)
    return best % MOD

for arr, want in examples:
    print(max_product_iter(build_tree(arr)), "| mong đợi", want)

110 | mong đợi 110
90 | mong đợi 90


Code nộp LeetCode (Cách 4):

In [14]:
class SolutionIter:
    def maxProduct(self, root: Optional[TreeNode]) -> int:
        order, stack = [], [root]
        while stack:
            node = stack.pop()
            if node:
                order.append(node)
                stack.append(node.left)
                stack.append(node.right)
        sub = {None: 0}
        for node in reversed(order):
            sub[node] = node.val + sub[node.left] + sub[node.right]
        total = sub[root]
        best = 0
        for node in order:
            p = sub[node] * (total - sub[node])
            if p > best:
                best = p
        return best % (10**9 + 7)

print([SolutionIter().maxProduct(build_tree(a)) for a, _ in examples])

[110, 90]


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (`order`, `sub`).

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 2** `[1,null,2,3,4,null,null,5,6]` (`total = 21`):

```
1
 \
  2
 / \
3   4
   / \
  5   6
```

| thứ tự hậu thứ tự | nút | s = tổng cây con | s·(21 − s) | best |
|---|---|---|---|---|
| 1 | 3 | 3 | 54 | 54 |
| 2 | 5 | 5 | 80 | 80 |
| 3 | 6 | 6 | 90 | 90 |
| 4 | 4 | 15 | 90 | 90 |
| 5 | 2 | 20 | 20 | 90 |
| 6 | 1 | 21 | 0 | 90 |

In [15]:
def trace_product(arr):
    root = build_tree(arr)
    total = subtree_sum(root)
    print("total =", total)
    best = 0
    step = 0

    def dfs(node, depth):
        nonlocal best, step
        if node is None:
            return 0
        s = node.val + dfs(node.left, depth + 1) + dfs(node.right, depth + 1)
        prod = s * (total - s)
        step += 1
        old = best
        best = max(best, prod)
        print(f"{'   ' * depth}bước {step}: nút {node.val}, s={s}, tích={s}*{total - s}={prod}, best {old} -> {best}")
        return s

    dfs(root, 0)
    print("đáp án =", best % MOD)

trace_product(examples[1][0])

total = 21
      bước 1: nút 3, s=3, tích=3*18=54, best 0 -> 54
         bước 2: nút 5, s=5, tích=5*16=80, best 54 -> 80
         bước 3: nút 6, s=6, tích=6*15=90, best 80 -> 90
      bước 4: nút 4, s=15, tích=15*6=90, best 90 -> 90
   bước 5: nút 2, s=20, tích=20*1=20, best 90 -> 90
bước 6: nút 1, s=21, tích=21*0=0, best 90 -> 90
đáp án = 90


## 9. Code nộp LeetCode

Bản chọn nộp: Cách 4 (ngăn xếp) — `O(n)`, không lo giới hạn đệ quy với cây lệch 5·10⁴ nút.

In [16]:
class Solution:
    def maxProduct(self, root: Optional[TreeNode]) -> int:
        MOD = 10**9 + 7
        order, stack = [], [root]
        while stack:                                   # tiền thứ tự
            node = stack.pop()
            if node:
                order.append(node)
                stack.append(node.left)
                stack.append(node.right)
        sub = {None: 0}
        for node in reversed(order):                   # ngược lại = con trước cha
            sub[node] = node.val + sub[node.left] + sub[node.right]
        total = sub[root]
        best = max(s * (total - s) for s in sub.values())
        return best % MOD                              # max TRƯỚC, mod SAU

for arr, want in examples:
    got = Solution().maxProduct(build_tree(arr))
    print(got, "| mong đợi", want, "| OK" if got == want else "| SAI")

110 | mong đợi 110 | OK
90 | mong đợi 90 | OK


## 10. Vì sao thuật toán đúng

1. **Tương ứng cạnh ↔ nút:** mỗi cạnh của cây nối một nút `v ≠ root` với cha của nó, và ngược lại mỗi nút `v ≠ root` có đúng một cạnh lên cha. Xoá cạnh đó ⇒ đúng hai thành phần: cây con gốc `v` (tổng `sub(v)`) và phần còn lại (tổng `total − sub(v)`, vì mọi giá trị thuộc đúng một trong hai phần).
2. **Tính đúng `sub(v)`:** hậu thứ tự đảm bảo khi xử lý `v`, tổng hai cây con đã có ⇒ `sub(v) = val + sub(left) + sub(right)` (quy nạp theo chiều cao).
3. **Xét đủ mọi lựa chọn:** thuật toán lấy `max` của `sub(v)·(total − sub(v))` trên **mọi** nút; các nút không phải gốc là đúng tập các phương án hợp lệ, còn gốc cho tích `0` (và mọi phương án hợp lệ có tích `≥ 1·1 > 0` vì giá trị `≥ 1`) nên không làm sai `max`.
4. **Mod đúng chỗ:** `max` được tính trên số nguyên chính xác; chỉ kết quả cuối được lấy `% (10^9+7)` — đúng yêu cầu "maximize before mod".

## 11. Phân tích Time / Space

Đặt `n` = số nút (`≤ 5·10^4`), `h` = chiều cao.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Cắt từng cạnh | `O(n²)` | `O(n)` | quá chậm với `n` lớn |
| 2. Hai lượt DFS đệ quy | `O(n)` | `O(h)` | gọn, cần đệ quy đủ sâu |
| 3. Một lượt + list tổng | `O(n)` | `O(n)` | dễ mở rộng |
| 4. Ngăn xếp | `O(n)` | `O(n)` | an toàn với cây lệch |

Số nguyên: tích tối đa ≈ `6·10^16` — Python xử lý chính xác; trong C++/Java phải dùng `long long`/`long`.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
sys.setrecursionlimit(200000)      # để các cách đệ quy chạy được cây lệch trong test

solvers = {
    "brute": max_product_brute,
    "two_pass": max_product_two_pass,
    "one_pass": max_product_one_pass,
    "closest_half": max_product_closest_half,
    "iter": max_product_iter,
    "submit": lambda r: Solution().maxProduct(r),
    "submit_two": lambda r: SolutionTwoPass().maxProduct(r),
    "submit_one": lambda r: SolutionOnePass().maxProduct(r),
    "submit_iter": lambda r: SolutionIter().maxProduct(r),
}
cases = examples + [
    ([1, 1], 1),                                        # 2 nút
    ([2, 3, 9, 10, 7, 8, 6, 5, 4, 11, 1], 1025),
    ([1, 1, None, 1, None, 1], 4),                      # lệch trái 4 nút: tổng 4 -> 2*2
    ([10**4, 10**4], 10**8),
    ([5, 1, 1], 6),                                     # cắt 1 nút: 1*6
]
for arr, want in cases:
    for name, f in solvers.items():
        assert f(build_tree(arr)) == want, (name, arr, f(build_tree(arr)), want)
print("Tất cả", len(cases), "case qua cho", len(solvers), "lời giải")

Tất cả 7 case qua cho 9 lời giải


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Giá trị nút lớn (tới `10^4`) và cây vài trăm nút để tích **vượt** `10^9 + 7` — kiểm tra luôn việc mod đúng chỗ.

In [18]:
def random_tree(rng, n, vmax):
    root = TreeNode(rng.randint(1, vmax))
    nodes = [root]
    while len(nodes) < n:
        p = rng.choice(nodes)
        side = rng.choice(["left", "right"])
        if getattr(p, side) is None:
            ch = TreeNode(rng.randint(1, vmax))
            setattr(p, side, ch)
            nodes.append(ch)
    return tree_to_list(root)

rng = random.Random(1339)
big_products = 0
for _ in range(400):
    arr = random_tree(rng, rng.randint(2, 60), rng.choice([10, 10**4]))
    want = max_product_brute(build_tree(arr))
    for name, f in solvers.items():
        assert f(build_tree(arr)) == want, (name, arr)
    t = sum(v for v in arr if v is not None)
    big_products += (t // 2) * (t - t // 2) >= MOD
print("400 cây ngẫu nhiên: mọi cách khớp brute force;", big_products, "cây có tích có thể vượt MOD")

400 cây ngẫu nhiên: mọi cách khớp brute force; 169 cây có tích có thể vượt MOD


### 12.3 Cây lớn nhất: 5·10⁴ nút (cân bằng và lệch)

In [19]:
rng = random.Random(7)
n = 5 * 10**4
balanced = [rng.randint(1, 10**4) for _ in range(n)]
skewed = [10**4]
for _ in range(n - 1):
    skewed += [None, 10**4]                 # lệch phải: mỗi nút chỉ có con phải
for label, arr in [("cân bằng", balanced), ("lệch phải", skewed)]:
    out = {}
    for name in ["submit", "submit_iter", "submit_two", "one_pass"]:
        root = build_tree(arr)
        t0 = time.time()
        out[name] = solvers[name](root)
        out[name + "_t"] = (time.time() - t0) * 1000
    assert len({out[k] for k in ["submit", "submit_iter", "submit_two", "one_pass"]}) == 1
    print(f"{label:9}: đáp án {out['submit']} | " + ", ".join(f"{k} {out[k + '_t']:.0f}ms" for k in ["submit", "submit_iter", "submit_two", "one_pass"]))

cân bằng : đáp án 639130272 | submit 23ms, submit_iter 28ms, submit_two 20ms, one_pass 14ms


lệch phải: đáp án 562500007 | submit 24ms, submit_iter 25ms, submit_two 31ms, one_pass 21ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Mod từng tích rồi mới lấy max

In [20]:
def wrong_mod_first(root):
    total, sums = all_subtree_sums(root)
    return max(s * (total - s) % MOD for s in sums)       # SAI: max của phần dư

rng = random.Random(11)
for _ in range(1000):
    arr = random_tree(rng, rng.randint(20, 200), 10**4)
    r = build_tree(arr)
    if wrong_mod_first(r) != Solution().maxProduct(r):
        break
total, sums = all_subtree_sums(build_tree(arr))
best_true = max(sums, key=lambda s: s * (total - s))
best_mod = max(sums, key=lambda s: s * (total - s) % MOD)
print("sai :", wrong_mod_first(build_tree(arr)), "| đúng:", Solution().maxProduct(build_tree(arr)))
print(f"tích thật lớn nhất {best_true * (total - best_true)} (s={best_true}) — "
      f"nhưng tích có phần dư lớn nhất là {best_mod * (total - best_mod)} (s={best_mod})")

sai : 989241474 | đúng: 72657440
tích thật lớn nhất 125072658315 (s=332981) — nhưng tích có phần dư lớn nhất là 15989241579 (s=23333)


**Vì sao sai:** phép mod **không bảo toàn thứ tự**: số lớn hơn có thể có phần dư nhỏ hơn. Đề yêu cầu tối đa hoá **tích thật**, rồi mới mod. **Sửa:** `max(...)` trên số nguyên đầy đủ, `% MOD` ở cuối.

### Lỗi 2 — Chỉ thử cắt các cạnh kề gốc

In [21]:
def wrong_root_edges_only(root):
    total = subtree_sum(root)
    best = 0
    for child in (root.left, root.right):
        if child:
            s = subtree_sum(child)
            best = max(best, s * (total - s))
    return best % MOD

arr, want = examples[1]
print("sai :", wrong_root_edges_only(build_tree(arr)), "| đúng:", Solution().maxProduct(build_tree(arr)))

sai : 20 | đúng: 90


**Vì sao sai:** cạnh tốt nhất có thể nằm sâu bên dưới. Ở Ví dụ 2, gốc chỉ có một con (`2`, tổng 20) ⇒ tích `20·1 = 20`, trong khi cắt cạnh `2–4` (hoặc `4–6`) cho `15·6 = 90`. **Sửa:** xét **mọi** nút không phải gốc.

### Lỗi 3 — Chỉ xét tổng cây con `≤ total/2`

In [22]:
def wrong_only_small_half(root):
    total, sums = all_subtree_sums(root)
    cands = [s for s in sums if 2 * s <= total]          # SAI: bỏ các cây con "nặng"
    return max((s * (total - s) for s in cands), default=0) % MOD

for arr in ([1, 10], [3, 7, None, 1, 1]):
    print(arr, "| sai:", wrong_only_small_half(build_tree(arr)), "| đúng:", Solution().maxProduct(build_tree(arr)))

[1, 10] | sai: 0 | đúng: 10
[3, 7, None, 1, 1] | sai: 11 | đúng: 27


**Vì sao sai:** lý luận "chỉ cần phần nhỏ hơn" quên rằng **phần còn lại không phải lúc nào cũng là một cây con** có trong danh sách. Với `[1,10]`, cây con duy nhất có tổng `10 > 11/2`; phần kia (`1`, chỉ gồm gốc) không phải tổng cây con ⇒ bị bỏ sót, trả `0`. **Sửa:** xét **mọi** tổng cây con (hoặc chọn theo `|2s − total|` nhỏ nhất như `max_product_closest_half`).

### Lỗi 4 — Đệ quy quá sâu với cây lệch

In [23]:
def recursive_sum(node):
    return 0 if node is None else node.val + recursive_sum(node.left) + recursive_sum(node.right)

old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
deep = build_tree([1] + [None, 1] * 4999)        # lệch phải 5000 nút
try:
    recursive_sum(deep)
    print("không lỗi")
except RecursionError as e:
    print("đệ quy văng lỗi:", type(e).__name__, "với cây lệch 5000 nút")
finally:
    sys.setrecursionlimit(old)
print("ngăn xếp vẫn chạy:", Solution().maxProduct(deep))

đệ quy văng lỗi: RecursionError với cây lệch 5000 nút
ngăn xếp vẫn chạy: 6250000


**Vì sao sai:** đệ quy sâu bằng chiều cao cây; cây lệch 5·10⁴ nút vượt xa giới hạn ~1000 mặc định. **Sửa:** dùng ngăn xếp (Cách 4) hoặc nâng `sys.setrecursionlimit` khi chắc chắn môi trường cho phép.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Xoá một cạnh** của cây, tách làm hai" ⇒ mỗi cạnh ↔ một cây con ⇒ cần **tổng/size mọi cây con** (một lượt hậu thứ tự).
- Phần còn lại = **tổng toàn cây − cây con** — không cần duyệt lại.
- "Maximize then mod" ⇒ trong Python cứ tính chính xác rồi mod cuối; trong ngôn ngữ khác dùng kiểu 64-bit.
- Tích của hai số có tổng cố định lớn nhất khi hai số **gần nhau nhất**.
- Cây tới `10^4–10^5` nút ⇒ cân nhắc duyệt **bằng ngăn xếp** trong Python.

## 15. Biến thể của bài

1. **Cân bằng nhất** (Bài tập 5): xoá một cạnh để `|tổng1 − tổng2|` nhỏ nhất.
2. **LeetCode 663 — Equal Tree Partition** (Premium): có thể tách thành hai phần **bằng nhau** không? ⇒ có cây con (khác gốc) tổng `total/2`.
3. **LeetCode 2049 — Count Nodes With the Highest Score:** xoá một **nút** (tách thành tối đa 3 phần) — dùng **kích thước** cây con.
4. **LeetCode 2322 — Minimum Score After Removals on a Tree:** xoá **hai** cạnh, dùng XOR cây con.
5. Đồ thị dạng cây cho bằng danh sách cạnh: cùng ý tưởng, DFS trên danh sách kề.

In [24]:
def equal_partition(root):
    """LeetCode 663: tách được hai phần tổng bằng nhau?"""
    sums = []
    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        sums.append(s)
        return s
    total = dfs(root)
    sums.pop()                                  # bỏ tổng của chính gốc
    return total % 2 == 0 and total // 2 in sums

print(equal_partition(build_tree([5, 10, 10, None, None, 2, 3])), equal_partition(build_tree([1, 2, 10, None, None, 2, 20])))

True False


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cắt từng cạnh | cắt, tính lại hai tổng, nối lại | `O(n²)` | `O(n)` | kiểm chứng, `n` nhỏ |
| 2. Hai lượt DFS | lượt 1 tổng, lượt 2 cập nhật max | `O(n)` | `O(h)` | gọn, phổ biến |
| 3. Một lượt + list | lưu mọi tổng cây con rồi max | `O(n)` | `O(n)` | dễ mở rộng (gần `total/2`, tách bằng nhau) |
| 4. Ngăn xếp | tiền thứ tự rồi xử lý ngược | `O(n)` | `O(n)` | cây sâu, Python an toàn |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`maxProduct([1,1])` và `maxProduct([5,1,1])` trả về gì?

### Bài 2 (tính tay)
`root = [2,3,9,10,7,8,6,5,4,11,1]`. Tính tổng toàn cây, liệt kê tổng các cây con, tìm đáp án.

### Bài 3 (điền TODO)
Hoàn thiện `max_product_todo` theo Cách 2.

### Bài 4 (tự viết lại)
Viết lại Cách 4 nhưng **không dùng dict**: gán thẳng tổng cây con vào thuộc tính tạm của nút hoặc dùng list song song — đặt tên `max_product_ex4`.

### Bài 5 (mini challenge)
Viết `min_diff_split(root)` trả về `min |tổng1 − tổng2|` khi xoá đúng một cạnh. Kiểm chứng bằng brute force cắt từng cạnh.

In [25]:
# Bài 3, 4, 5 — khung (chỉ định nghĩa, không gọi)
def max_product_todo(root):
    total = None            # TODO: tổng toàn cây
    best = 0
    def dfs(node):
        # TODO: trả tổng cây con, cập nhật best
        pass
    # TODO: gọi dfs và trả best % MOD
    pass

def max_product_ex4_todo(root):
    # TODO
    pass

def min_diff_split_todo(root):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Chỉ có 1 cạnh ⇒ `1·1`. (2) `[5,1,1]`: tổng 7, cây con 1 và 1. (3) `1·6 = 6`.

**Bài 2:** (1) Tổng 66. (2) Tổng cây con (sắp tăng): `1,4,5,6,8,11,19,19,23,41,66`. (3) Gần 33 nhất là `41` hoặc `23` ⇒ `41·25 = 1025`, `23·43 = 989` ⇒ 1025.

**Bài 3:** (1) `total = subtree_sum(root)`. (2) Trong `dfs`: `s = val + dfs(left) + dfs(right)`; `best = max(best, s*(total-s))` (dùng `nonlocal`). (3) `return best % MOD`.

**Bài 4:** (1) `order` tiền thứ tự. (2) `idx = {node: i}` hoặc gán `node.sub`. (3) Duyệt ngược, cộng tổng con.

**Bài 5:** (1) Hai phần có tổng `s` và `total − s` ⇒ hiệu `|total − 2s|`. (2) Lấy `min` trên mọi nút **không phải gốc**. (3) Brute: cắt từng cạnh như Cách 1.

## 19. Đáp án tham khảo

In [26]:
# Bài 1, 2
print("Bài 1:", Solution().maxProduct(build_tree([1, 1])), Solution().maxProduct(build_tree([5, 1, 1])))
assert Solution().maxProduct(build_tree([1, 1])) == 1 and Solution().maxProduct(build_tree([5, 1, 1])) == 6
t2, s2 = all_subtree_sums(build_tree([2, 3, 9, 10, 7, 8, 6, 5, 4, 11, 1]))
print("Bài 2: total =", t2, "| tổng cây con =", sorted(s2), "| đáp án =", Solution().maxProduct(build_tree([2, 3, 9, 10, 7, 8, 6, 5, 4, 11, 1])))
assert t2 == 66 and Solution().maxProduct(build_tree([2, 3, 9, 10, 7, 8, 6, 5, 4, 11, 1])) == 1025

# Bài 3
def max_product_sol(root):
    total = subtree_sum(root)
    best = 0
    def dfs(node):
        nonlocal best
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        best = max(best, s * (total - s))
        return s
    dfs(root)
    return best % MOD

# Bài 4
def max_product_ex4(root):
    order, stack = [], [root]
    while stack:
        nd = stack.pop()
        if nd:
            order.append(nd)
            stack.append(nd.left)
            stack.append(nd.right)
    idx = {nd: i for i, nd in enumerate(order)}
    sub = [0] * len(order)
    for i in range(len(order) - 1, -1, -1):
        nd = order[i]
        sub[i] = nd.val + (sub[idx[nd.left]] if nd.left else 0) + (sub[idx[nd.right]] if nd.right else 0)
    total = sub[0]
    return max(s * (total - s) for s in sub) % MOD

for arr, want in cases:
    assert max_product_sol(build_tree(arr)) == want and max_product_ex4(build_tree(arr)) == want
print("Bài 1–4: qua")

Bài 1: 1 6
Bài 2: total = 66 | tổng cây con = [1, 4, 5, 6, 8, 11, 19, 19, 23, 41, 66] | đáp án = 1025
Bài 1–4: qua


In [27]:
# Bài 5
def min_diff_split(root):
    total, sums = all_subtree_sums(root)
    sums.pop()                                   # tổng của gốc nằm cuối (hậu thứ tự) -> bỏ
    return min(abs(total - 2 * s) for s in sums)

def min_diff_brute(root):
    def tsum(node):
        return 0 if node is None else node.val + tsum(node.left) + tsum(node.right)
    best = None
    stack = [root]
    while stack:
        p = stack.pop()
        for side in ("left", "right"):
            c = getattr(p, side)
            if c:
                setattr(p, side, None)
                d = abs(tsum(root) - tsum(c))
                setattr(p, side, c)
                best = d if best is None else min(best, d)
                stack.append(c)
    return best

print("Bài 5:", min_diff_split(build_tree(examples[0][0])), min_diff_split(build_tree(examples[1][0])))
rng = random.Random(5)
for _ in range(300):
    arr = random_tree(rng, rng.randint(2, 40), 50)
    assert min_diff_split(build_tree(arr)) == min_diff_brute(build_tree(arr))
print("Bài 5: 300 cây ngẫu nhiên khớp brute force")

Bài 5: 1 9
Bài 5: 300 cây ngẫu nhiên khớp brute force


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tổng mọi cây con" bằng ngăn xếp:**

```python
order, stack = [], [root]
while stack:
    nd = stack.pop()
    if nd:
        order.append(nd); stack.append(nd.left); stack.append(nd.right)
sub = {None: 0}
for nd in reversed(order):                 # con trước cha
    sub[nd] = nd.val + sub[nd.left] + sub[nd.right]
total = sub[root]
ans = max(s * (total - s) for s in sub.values()) % (10**9 + 7)
```

**Checklist trước khi submit:**
- [ ] Xét **mọi** nút (mọi cạnh), không chỉ cạnh kề gốc.
- [ ] Phần còn lại = `total − sub` (không chỉ xét `sub ≤ total/2`).
- [ ] Lấy `max` **trước**, `% (10**9 + 7)` **sau**.
- [ ] Cây sâu ⇒ ngăn xếp hoặc tăng giới hạn đệ quy.
- [ ] Đúng chữ ký: `maxProduct(self, root: Optional[TreeNode]) -> int`.